Install & Import Libraries

In [1]:
# Install required libraries
!pip install pandas scikit-learn matplotlib seaborn

# Import everything we need
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

print("All libraries imported successfully")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


All libraries imported successfully


Load All 3 Datasets

In [4]:
# Load the three structured log files
hdfs_df    = pd.read_csv(r'D:\Downloads\Capstone\NLP_Project\Dataset\Collected\HDFS_2k.log_structured.csv')
openssh_df = pd.read_csv(r'D:\Downloads\Capstone\NLP_Project\Dataset\Collected\Linux_2k.log_structured.csv')
linux_df   = pd.read_csv(r'D:\Downloads\Capstone\NLP_Project\Dataset\Collected\OpenSSH_2k.log_structured.csv')

# Check what columns each file has
print("HDFS columns:   ", list(hdfs_df.columns))
print("OpenSSH columns:", list(openssh_df.columns))
print("Linux columns:  ", list(linux_df.columns))

print(f"\nHDFS rows:    {len(hdfs_df)}")
print(f"OpenSSH rows: {len(openssh_df)}")
print(f"Linux rows:   {len(linux_df)}")

HDFS columns:    ['LineId', 'Date', 'Time', 'Pid', 'Level', 'Component', 'Content', 'EventId', 'EventTemplate']
OpenSSH columns: ['LineId', 'Month', 'Date', 'Time', 'Level', 'Component', 'PID', 'Content', 'EventId', 'EventTemplate']
Linux columns:   ['LineId', 'Date', 'Day', 'Time', 'Component', 'Pid', 'Content', 'EventId', 'EventTemplate']

HDFS rows:    2000
OpenSSH rows: 2000
Linux rows:   2000


Preview Each Dataset

In [5]:
# See a sample of each dataset
print("=== HDFS Sample ===")
print(hdfs_df[['Content', 'EventId', 'EventTemplate']].head(3).to_string())

print("\n=== OpenSSH Sample ===")
print(openssh_df[['Content', 'EventId', 'EventTemplate']].head(3).to_string())

print("\n=== Linux Sample ===")
print(linux_df[['Content', 'EventId', 'EventTemplate']].head(3).to_string())

=== HDFS Sample ===
                                                                                                                     Content EventId                                                                             EventTemplate
0                                                              PacketResponder 1 for block blk_38865049064139660 terminating     E10                                         PacketResponder <*> for block blk_<*> terminating
1                                                           PacketResponder 0 for block blk_-6952295868487656571 terminating     E10                                         PacketResponder <*> for block blk_<*> terminating
2  BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864      E6  BLOCK* NameSystem.addStoredBlock: blockMap updated: <*>:<*> is added to blk_<*> size <*>

=== OpenSSH Sample ===
                                                                

In [7]:
import pandas as pd
import re
import json
import os
from sklearn.model_selection import train_test_split

# =========================
# 0. Create output folder
# =========================
output_dir = "logBert_output"
os.makedirs(output_dir, exist_ok=True)



In [8]:
print(f"Output folder ready: {output_dir}")

# =========================
# 1. Load structured HDFS logs
# =========================

print("Shape:", hdfs_df.shape)
print("Columns:", hdfs_df.columns.tolist())
print(hdfs_df.head())


# =========================
# 2. Keep needed columns
# =========================
logbert_df = hdfs_df[['Content', 'EventId', 'EventTemplate']].copy()
logbert_df = logbert_df.dropna(subset=['Content', 'EventId', 'EventTemplate']).reset_index(drop=True)

print("\nRows after cleaning:", len(logbert_df))
print("Unique EventIds:", logbert_df['EventId'].nunique())


Output folder ready: logBert_output
Shape: (2000, 9)
Columns: ['LineId', 'Date', 'Time', 'Pid', 'Level', 'Component', 'Content', 'EventId', 'EventTemplate']
   LineId   Date    Time  Pid Level                     Component  \
0       1  81109  203615  148  INFO  dfs.DataNode$PacketResponder   
1       2  81109  203807  222  INFO  dfs.DataNode$PacketResponder   
2       3  81109  204005   35  INFO              dfs.FSNamesystem   
3       4  81109  204015  308  INFO  dfs.DataNode$PacketResponder   
4       5  81109  204106  329  INFO  dfs.DataNode$PacketResponder   

                                             Content EventId  \
0  PacketResponder 1 for block blk_38865049064139...     E10   
1  PacketResponder 0 for block blk_-6952295868487...     E10   
2  BLOCK* NameSystem.addStoredBlock: blockMap upd...      E6   
3  PacketResponder 2 for block blk_82291938032499...     E10   
4  PacketResponder 2 for block blk_-6670958622368...     E10   

                                       Even

In [9]:


# =========================
# 3. Extract BlockId from Content
# =========================
def extract_block_id(text):
    if pd.isna(text):
        return None
    match = re.search(r'(blk_-?\d+)', str(text))
    return match.group(1) if match else None

logbert_df['BlockId'] = logbert_df['Content'].apply(extract_block_id)

print("\nBlockId missing count:", logbert_df['BlockId'].isna().sum())
print(logbert_df[['Content', 'BlockId']].head())


# =========================
# 4. Keep rows with BlockId
# =========================
logbert_df = logbert_df.dropna(subset=['BlockId']).reset_index(drop=True)

print("\nRows after keeping BlockId:", len(logbert_df))
print("Unique BlockIds:", logbert_df['BlockId'].nunique())


# =========================
# 5. Build block-based event sequences
# =========================
session_df = (
    logbert_df
    .groupby('BlockId')['EventId']
    .apply(list)
    .reset_index()
)

session_df['sequence_length'] = session_df['EventId'].apply(len)
session_df['event_sequence'] = session_df['EventId'].apply(lambda x: ' '.join(map(str, x)))

print("\nTotal sessions:", len(session_df))
print(session_df.head())


# =========================
# 6. Save full session file
# =========================
session_df.to_csv(os.path.join(output_dir, "hdfs_logbert_sessions.csv"), index=False)
print("\nSaved: hdfs_logbert_sessions.csv")




BlockId missing count: 0
                                             Content                   BlockId
0  PacketResponder 1 for block blk_38865049064139...     blk_38865049064139660
1  PacketResponder 0 for block blk_-6952295868487...  blk_-6952295868487656571
2  BLOCK* NameSystem.addStoredBlock: blockMap upd...   blk_7128370237687728475
3  PacketResponder 2 for block blk_82291938032499...   blk_8229193803249955061
4  PacketResponder 2 for block blk_-6670958622368...  blk_-6670958622368987959

Rows after keeping BlockId: 2000
Unique BlockIds: 1994

Total sessions: 1994
                    BlockId EventId  sequence_length event_sequence
0  blk_-1030832046197982436    [E8]                1             E8
1  blk_-1046472716157313227    [E9]                1             E9
2  blk_-1049340855430710153   [E10]                1            E10
3  blk_-1055254430948037872    [E6]                1             E6
4  blk_-1067234447809438340   [E10]                1            E10

Saved: hdfs_l

In [10]:

# =========================
# 7. Split into train and test
# =========================
train_df, test_df = train_test_split(
    session_df,
    test_size=0.2,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTrain sessions:", len(train_df))
print("Test sessions:", len(test_df))


# =========================
# 8. Save train/test CSV files
# =========================
train_df.to_csv(os.path.join(output_dir, "hdfs_logbert_train_sessions.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "hdfs_logbert_test_sessions.csv"), index=False)

print("Saved: hdfs_logbert_train_sessions.csv")
print("Saved: hdfs_logbert_test_sessions.csv")


# =========================
# 9. Save train/test TXT files for LogBERT
# =========================
with open(os.path.join(output_dir, "hdfs_logbert_train.txt"), "w", encoding="utf-8") as f:
    for line in train_df['event_sequence']:
        f.write(line + "\n")

with open(os.path.join(output_dir, "hdfs_logbert_test.txt"), "w", encoding="utf-8") as f:
    for line in test_df['event_sequence']:
        f.write(line + "\n")

print("Saved: hdfs_logbert_train.txt")
print("Saved: hdfs_logbert_test.txt")



Train sessions: 1595
Test sessions: 399
Saved: hdfs_logbert_train_sessions.csv
Saved: hdfs_logbert_test_sessions.csv
Saved: hdfs_logbert_train.txt
Saved: hdfs_logbert_test.txt


In [11]:


# =========================
# 10. Build and save vocabulary
# =========================
unique_events = sorted(logbert_df['EventId'].astype(str).unique())
event2idx = {event: idx + 1 for idx, event in enumerate(unique_events)}

with open(os.path.join(output_dir, "hdfs_logbert_event_vocab.json"), "w", encoding="utf-8") as f:
    json.dump(event2idx, f, indent=2)

print("Saved: hdfs_logbert_event_vocab.json")
print("Vocabulary size:", len(event2idx))

# =========================
# 11. Also save cleaned base dataframe
# =========================
logbert_df.to_csv(os.path.join(output_dir, "hdfs_logbert_base.csv"), index=False)
print("Saved: hdfs_logbert_base.csv")

# =========================
# 12. Preview example outputs
# =========================
print("\nExample train sequence:")
print(train_df['event_sequence'].iloc[0])

print("\nExample vocab items:")
print(list(event2idx.items())[:10])

Saved: hdfs_logbert_event_vocab.json
Vocabulary size: 14
Saved: hdfs_logbert_base.csv

Example train sequence:
E7

Example vocab items:
[('E1', 1), ('E10', 2), ('E11', 3), ('E12', 4), ('E13', 5), ('E14', 6), ('E2', 7), ('E3', 8), ('E4', 9), ('E5', 10)]


Load Train, Test, Vocab

In [13]:
#import os
#import json

#output_dir = "logBert_output"

train_file = os.path.join(output_dir, "hdfs_logbert_train.txt")
test_file = os.path.join(output_dir, "hdfs_logbert_test.txt")
vocab_file = os.path.join(output_dir, "hdfs_logbert_event_vocab.json")

with open(train_file, "r", encoding="utf-8") as f:
    train_lines = [line.strip() for line in f if line.strip()]

with open(test_file, "r", encoding="utf-8") as f:
    test_lines = [line.strip() for line in f if line.strip()]

with open(vocab_file, "r", encoding="utf-8") as f:
    event2idx = json.load(f)

print("Train sequences:", len(train_lines))
print("Test sequences:", len(test_lines))
print("Vocab size:", len(event2idx))
print("Example train line:", train_lines[0])

Train sequences: 1595
Test sequences: 399
Vocab size: 14
Example train line: E7


convert sequence text into token IDs

In [14]:
def encode_sequence(seq, vocab):
    tokens = seq.split()
    return [vocab[token] for token in tokens if token in vocab]

train_encoded = [encode_sequence(seq, event2idx) for seq in train_lines]
test_encoded = [encode_sequence(seq, event2idx) for seq in test_lines]

print("Example encoded train sequence:")
print(train_encoded[0])

Example encoded train sequence:
[12]


In [15]:
# pad sequences to equal length

from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = max(len(seq) for seq in train_encoded + test_encoded)

X_train = pad_sequences(train_encoded, maxlen=max_len, padding='post', truncating='post')
X_test = pad_sequences(test_encoded, maxlen=max_len, padding='post', truncating='post')

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Example padded sequence:", X_train[0])

X_train shape: (1595, 2)
X_test shape: (399, 2)
Example padded sequence: [12  0]


In [17]:
# create PyTorch dataset

import torch
from torch.utils.data import Dataset, DataLoader

class LogDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = torch.tensor(sequences, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

train_dataset = LogDataset(X_train)
test_dataset = LogDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 50
Test batches: 13


In [18]:
# check one batch

for batch in train_loader:
    print("Batch shape:", batch.shape)
    print(batch[:2])
    break

Batch shape: torch.Size([32, 2])
tensor([[5, 0],
        [5, 0]])


In [19]:
print(session_df['sequence_length'].describe())
print(session_df['sequence_length'].value_counts().head(20))

count    1994.000000
mean        1.003009
std         0.054786
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: sequence_length, dtype: float64
sequence_length
1    1988
2       6
Name: count, dtype: int64
